# DeepSeek-V3: Full Architecture Replication

This notebook replicates the architecture of **DeepSeek-V3** — a 671B parameter
Mixture-of-Experts language model that achieves GPT-4 level performance while being
trained for only **$5.5M** (10x cheaper than comparable models).

DeepSeek-V3 combines nearly every technique covered in this repo into one system:
- **Multi-head Latent Attention (MLA)** — compresses KV cache for efficient inference
- **DeepSeekMoE** — 256 fine-grained experts + shared experts
- **Auxiliary-loss-free load balancing** — better than standard MoE balancing
- **Multi-Token Prediction (MTP)** — predicts multiple future tokens
- **FP8 mixed precision** — aggressive quantization during training
- **GRPO alignment** — RL without a critic (see `04_Reinforcement_Learning/03_GRPO`)

We build a scaled-down working version from scratch.

**Reference:** DeepSeek-AI (2024). *DeepSeek-V3 Technical Report.*
https://arxiv.org/abs/2412.19437

In [15]:
import sys, os
# In Colab, clone the repo so local imports (src/) work
if "google.colab" in str(get_ipython()):
    if not os.path.exists("/content/DeepLearning_101"):
        !git clone --depth 1 https://github.com/MaxiRuess/DeepLearning_101.git /content/DeepLearning_101
    os.chdir("/content/DeepLearning_101/notebooks/07_Model_Replications")
    sys.path.insert(0, "/content/DeepLearning_101")
else:
    sys.path.insert(0, os.path.abspath("../.."))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import matplotlib.pyplot as plt
from src.utils.device import set_seed

set_seed(42)
print(f"PyTorch version: {torch.__version__}")

Set seed for reproducibility: 42
PyTorch version: 2.11.0


## 1. Architecture at a Glance

| | DeepSeek-V3 | LLaMA 3 405B | GPT-4 (rumored) |
|---|---|---|---|
| Total params | 671B | 405B | ~1.8T |
| Active params/token | 37B | 405B (all) | ~220B (MoE) |
| Architecture | MoE + MLA | Dense + GQA | MoE (8 experts) |
| Attention | MLA (latent) | GQA | MHA |
| Experts | 256 routed + 1 shared | N/A (dense) | 8 large |
| Position encoding | RoPE (decoupled) | RoPE | Unknown |
| Training cost | **$5.5M** | ~$100M | ~$100M+ |
| Context length | 128K | 128K | 128K |

In [16]:
# DeepSeek-V3 parameter breakdown
config = {
    "layers": 61,
    "d_model": 7168,
    "n_heads": 128,
    "kv_latent_dim": 512,       # MLA compressed KV dimension
    "q_latent_dim": 1536,       # MLA compressed Q dimension
    "rope_head_dim": 64,        # decoupled RoPE dimension
    "n_routed_experts": 256,
    "n_shared_experts": 1,
    "top_k": 8,                 # 8 of 256 experts per token
    "expert_intermediate": 2048, # each expert's FFN inner dim
    "shared_intermediate": 2048,
    "vocab_size": 129280,
    "mtp_depth": 1,             # multi-token prediction depth
}

# Approximate parameter counts
d = config["d_model"]
n_layers = config["layers"]

# MLA attention params per layer
mla_params = (
    d * config["q_latent_dim"] +             # W_DQ (down-project Q)
    config["q_latent_dim"] * d +             # W_UQ (up-project Q)
    d * config["kv_latent_dim"] +            # W_DKV (down-project KV)
    config["kv_latent_dim"] * d +            # W_UK (up-project K)
    config["kv_latent_dim"] * d +            # W_UV (up-project V)
    d * config["rope_head_dim"] +            # W_QR (RoPE query)
    d * config["rope_head_dim"] +            # W_KR (RoPE key)
    d * d                                     # W_O (output projection)
)

# MoE params per layer
expert_params = 2 * d * config["expert_intermediate"]  # one expert FFN
routed_params = config["n_routed_experts"] * expert_params
shared_params = config["n_shared_experts"] * 2 * d * config["shared_intermediate"]
router_params = d * config["n_routed_experts"]
moe_params = routed_params + shared_params + router_params

# Active params per token
active_expert_params = config["top_k"] * expert_params + shared_params

total = n_layers * (mla_params + moe_params) + config["vocab_size"] * d
active = n_layers * (mla_params + active_expert_params) + config["vocab_size"] * d

print(f"DeepSeek-V3 Parameter Breakdown:")
print(f"  MLA attention per layer: {mla_params/1e6:.0f}M")
print(f"  MoE FFN per layer:       {moe_params/1e6:.0f}M (256 experts)")
print(f"  Active FFN per token:    {active_expert_params/1e6:.0f}M (top-8 + shared)")
print(f"  Embedding:               {config['vocab_size'] * d / 1e6:.0f}M")
print(f"\n  Total parameters:   {total/1e9:.1f}B")
print(f"  Active per token:   {active/1e9:.1f}B ({100*active/total:.0f}%)")

DeepSeek-V3 Parameter Breakdown:
  MLA attention per layer: 85M
  MoE FFN per layer:       7547M (256 experts)
  Active FFN per token:    264M (top-8 + shared)
  Embedding:               927M

  Total parameters:   466.5B
  Active per token:   22.3B (5%)


## 2. RMSNorm and SwiGLU

Before building the main components, we need two primitives that DeepSeek-V3 uses
instead of the original Transformer's LayerNorm and ReLU:

**RMSNorm** (simpler than LayerNorm — no mean subtraction):
$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}} \cdot \gamma$

**SwiGLU** (gated activation — better than ReLU for LLMs):
$\text{SwiGLU}(x) = \text{Swish}(xW_1) \odot xW_3$ then $\cdot W_2$

In [17]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (no mean subtraction)."""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class SwiGLU(nn.Module):
    """SwiGLU activation with gated linear unit."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)  # gate projection
        self.w3 = nn.Linear(d_model, d_ff, bias=False)  # up projection
        self.w2 = nn.Linear(d_ff, d_model, bias=False)  # down projection

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


# Demo
x = torch.randn(1, 4, 32)
print(f"RMSNorm: {RMSNorm(32)(x).shape}")
print(f"SwiGLU:  {SwiGLU(32, 64)(x).shape}")

RMSNorm: torch.Size([1, 4, 32])
SwiGLU:  torch.Size([1, 4, 32])


## 3. Multi-head Latent Attention (MLA)

**The KV cache problem:** In standard MHA, each layer stores the full K and V tensors
for every past token during inference. For 128K context with 128 heads:

$\text{KV cache per layer} = 2 \times \text{seq\_len} \times n_{heads} \times d_{head} \times 2\text{B}$

For DeepSeek-V3 scale, this is enormous.

**MLA's solution:** Compress K and V into a low-rank **latent vector** $c^{KV}$:

$c_t^{KV} = W^{DKV} h_t \quad (\text{down-project: } d_{model} \rightarrow d_c)$

At inference, only $c^{KV}$ is cached (much smaller than full K, V). K and V are
reconstructed on-the-fly:

$K_t = W^{UK} c_t^{KV}, \quad V_t = W^{UV} c_t^{KV}$

**Decoupled RoPE:** Position encoding is applied to a separate small vector that's
concatenated with the content-based key, not to the full key. This allows the
KV compression to be position-independent.

In [18]:
class MultiHeadLatentAttention(nn.Module):
    """
    Multi-head Latent Attention (MLA) from DeepSeek-V2/V3.
    Compresses KV cache via low-rank projection.
    """

    def __init__(self, d_model, n_heads, kv_latent_dim, q_latent_dim,
                 rope_head_dim, max_seq_len=4096, rope_base=10000.0):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.kv_latent_dim = kv_latent_dim
        self.rope_head_dim = rope_head_dim

        # Q path: down-project → up-project to multi-head
        self.W_DQ = nn.Linear(d_model, q_latent_dim, bias=False)
        self.W_UQ = nn.Linear(q_latent_dim, n_heads * self.head_dim, bias=False)

        # KV path: down-project to latent → up-project to K, V
        self.W_DKV = nn.Linear(d_model, kv_latent_dim, bias=False)
        self.W_UK = nn.Linear(kv_latent_dim, n_heads * self.head_dim, bias=False)
        self.W_UV = nn.Linear(kv_latent_dim, n_heads * self.head_dim, bias=False)

        # Decoupled RoPE projections (small, separate from content)
        self.W_QR = nn.Linear(d_model, n_heads * rope_head_dim, bias=False)
        self.W_KR = nn.Linear(d_model, rope_head_dim, bias=False)  # shared across heads

        # Output projection
        self.W_O = nn.Linear(n_heads * self.head_dim, d_model, bias=False)

        # Precompute RoPE frequencies
        freqs = 1.0 / (rope_base ** (torch.arange(0, rope_head_dim, 2).float() / rope_head_dim))
        positions = torch.arange(max_seq_len).float()
        angles = torch.outer(positions, freqs)
        self.register_buffer('rope_cos', angles.cos())
        self.register_buffer('rope_sin', angles.sin())

    def _apply_rope(self, x, seq_len):
        """Apply rotary embeddings to a tensor."""
        cos = self.rope_cos[:seq_len].unsqueeze(1)  # [N, 1, rope_dim//2] for head broadcast
        sin = self.rope_sin[:seq_len].unsqueeze(1)  # [N, 1, rope_dim//2]
        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]
        rotated = torch.stack([x_even * cos - x_odd * sin,
                               x_even * sin + x_odd * cos], dim=-1)
        return rotated.flatten(-2)

    def forward(self, x):
        B, N, _ = x.shape

        # Q: down-project → up-project → multi-head
        q_content = self.W_UQ(self.W_DQ(x)).view(B, N, self.n_heads, self.head_dim)

        # KV: down-project to latent (THIS is what gets cached at inference)
        c_kv = self.W_DKV(x)  # [B, N, kv_latent_dim] — the compressed cache

        # Up-project to K, V
        k_content = self.W_UK(c_kv).view(B, N, self.n_heads, self.head_dim)
        v = self.W_UV(c_kv).view(B, N, self.n_heads, self.head_dim)

        # Decoupled RoPE (separate position vectors)
        q_rope = self.W_QR(x).view(B, N, self.n_heads, self.rope_head_dim)
        k_rope = self.W_KR(x).view(B, N, 1, self.rope_head_dim).expand(-1, -1, self.n_heads, -1)

        q_rope = self._apply_rope(q_rope, N)
        k_rope = self._apply_rope(k_rope, N)

        # Concatenate content + position for Q and K
        q = torch.cat([q_content, q_rope], dim=-1)  # [B, N, heads, head_dim + rope_dim]
        k = torch.cat([k_content, k_rope], dim=-1)

        # Attention
        q = q.transpose(1, 2)  # [B, heads, N, dim]
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        full_dim = self.head_dim + self.rope_head_dim
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(full_dim)
        weights = torch.softmax(scores, dim=-1)
        out = weights @ v  # [B, heads, N, head_dim]

        # Concatenate heads and project
        out = out.transpose(1, 2).contiguous().view(B, N, self.n_heads * self.head_dim)
        return self.W_O(out)


# Demo
mla = MultiHeadLatentAttention(
    d_model=128, n_heads=4, kv_latent_dim=32,
    q_latent_dim=64, rope_head_dim=16
)
x = torch.randn(1, 16, 128)
out = mla(x)
print(f"MLA: {x.shape} → {out.shape}")
print(f"Parameters: {sum(p.numel() for p in mla.parameters()):,}")

MLA: torch.Size([1, 16, 128]) → torch.Size([1, 16, 128])
Parameters: 55,296


In [19]:
# KV cache size comparison: standard MHA vs MLA
d_model = 7168
n_heads = 128
head_dim = d_model // n_heads  # 56
kv_latent = 512
n_layers = 61

seq_lengths = [1024, 4096, 16384, 65536, 131072]

print(f"KV Cache Size Comparison (DeepSeek-V3 scale, {n_layers} layers, FP16):")
print(f"{'Seq len':>10s} {'Standard MHA':>15s} {'MLA':>15s} {'Savings':>10s}")

for seq in seq_lengths:
    # Standard: 2 (K,V) × seq × n_heads × head_dim × 2 bytes × n_layers
    mha_bytes = 2 * seq * n_heads * head_dim * 2 * n_layers
    # MLA: seq × kv_latent_dim × 2 bytes × n_layers (only cache the latent)
    mla_bytes = seq * kv_latent * 2 * n_layers
    savings = 1 - mla_bytes / mha_bytes
    print(f"{seq:>10,} {mha_bytes/1e9:>12.1f} GB {mla_bytes/1e9:>12.1f} GB {savings:>9.0%}")

KV Cache Size Comparison (DeepSeek-V3 scale, 61 layers, FP16):
   Seq len    Standard MHA             MLA    Savings
     1,024          1.8 GB          0.1 GB       96%
     4,096          7.2 GB          0.3 GB       96%
    16,384         28.7 GB          1.0 GB       96%
    65,536        114.6 GB          4.1 GB       96%
   131,072        229.2 GB          8.2 GB       96%


In [20]:
# Verify MLA output
mla_test = MultiHeadLatentAttention(
    d_model=64, n_heads=4, kv_latent_dim=16, q_latent_dim=32, rope_head_dim=8
)
x_test = torch.randn(2, 8, 64)  # batch=2, seq=8
out_test = mla_test(x_test)

print(f"Input:  {x_test.shape}")
print(f"Output: {out_test.shape}")
print(f"KV latent dim: {16} vs full KV dim: {64} → {16/64:.0%} compression")

Input:  torch.Size([2, 8, 64])
Output: torch.Size([2, 8, 64])
KV latent dim: 16 vs full KV dim: 64 → 25% compression


## 4. DeepSeekMoE

DeepSeek-V3's MoE differs from standard MoE (Mixtral) in two key ways:

**1. Fine-grained experts:** 256 small experts instead of 8 large ones. Each token
activates 8 of 256. More experts = more specialized knowledge per expert.

**2. Shared experts:** 1 expert that ALL tokens pass through, regardless of routing.
The shared expert captures common knowledge (syntax, grammar). Routed experts
capture specialized knowledge (math, code, languages).

$y = \text{SharedExpert}(x) + \sum_{i \in \text{TopK}} w_i \cdot \text{RoutedExpert}_i(x)$

In [21]:
class DeepSeekMoELayer(nn.Module):
    """
    DeepSeekMoE: shared expert(s) + fine-grained routed experts.
    """

    def __init__(self, d_model, expert_ff_dim, shared_ff_dim,
                 n_routed_experts=256, n_shared_experts=1, top_k=8):
        super().__init__()
        self.n_routed = n_routed_experts
        self.top_k = top_k

        # Shared expert(s) — always active
        self.shared_experts = nn.ModuleList([
            SwiGLU(d_model, shared_ff_dim) for _ in range(n_shared_experts)
        ])

        # Routed experts — top-K selected per token
        self.routed_experts = nn.ModuleList([
            SwiGLU(d_model, expert_ff_dim) for _ in range(n_routed_experts)
        ])

        # Router
        self.gate = nn.Linear(d_model, n_routed_experts, bias=False)

        # Auxiliary-loss-free bias (adjusted dynamically, not by gradient)
        self.register_buffer('expert_bias', torch.zeros(n_routed_experts))

    def forward(self, x):
        B, N, D = x.shape

        # Shared expert output (always computed)
        shared_out = sum(expert(x) for expert in self.shared_experts)

        # Router with bias-based load balancing
        router_logits = self.gate(x) + self.expert_bias  # bias adjusts routing
        top_k_logits, indices = torch.topk(router_logits, self.top_k, dim=-1)
        weights = torch.softmax(top_k_logits, dim=-1)

        # Compute routed expert outputs
        routed_out = torch.zeros_like(x)
        for k in range(self.top_k):
            expert_idx = indices[:, :, k]
            expert_weight = weights[:, :, k].unsqueeze(-1)
            for i in range(self.n_routed):
                mask = (expert_idx == i)
                if mask.any():
                    routed_out[mask] += (expert_weight[mask] *
                                        self.routed_experts[i](x[mask])).squeeze(-2)

        # Update bias for load balancing (no gradient — heuristic)
        if self.training:
            with torch.no_grad():
                flat = indices.reshape(-1)
                for i in range(self.n_routed):
                    count = (flat == i).float().sum()
                    target = flat.numel() / self.n_routed
                    # Decrease bias for overloaded, increase for underloaded
                    self.expert_bias[i] -= 0.001 * (count - target)

        return shared_out + routed_out, router_logits


# Demo (scaled down: 16 experts instead of 256)
ds_moe = DeepSeekMoELayer(
    d_model=64, expert_ff_dim=32, shared_ff_dim=64,
    n_routed_experts=16, n_shared_experts=1, top_k=4
)
x = torch.randn(1, 8, 64)
out, logits = ds_moe(x)
print(f"DeepSeekMoE: {x.shape} → {out.shape}")
print(f"Router: 16 experts, top-4 per token")
print(f"Shared expert always active + 4 routed experts per token")

DeepSeekMoE: torch.Size([1, 8, 64]) → torch.Size([1, 8, 64])
Router: 16 experts, top-4 per token
Shared expert always active + 4 routed experts per token


## 5. Auxiliary-Loss-Free Load Balancing

Standard MoE uses an auxiliary loss to balance expert loads (see `05_Papers/04_MoE`).
But this **hurts model quality** — the aux loss competes with the main training objective.

DeepSeek-V3's innovation: **bias-based balancing** (no auxiliary loss).

Each expert has a bias term $b_i$ added to its router score. During training:
- If expert $i$ is **overloaded**: decrease $b_i$ (fewer tokens routed to it)
- If expert $i$ is **underloaded**: increase $b_i$ (more tokens routed to it)

This adjustment is **not a gradient** — it's a simple heuristic update after each batch.
The main training loss is never contaminated by balancing concerns.

In [22]:
# Show the bias adjustment in action
ds_moe_demo = DeepSeekMoELayer(
    d_model=32, expert_ff_dim=16, shared_ff_dim=32,
    n_routed_experts=8, n_shared_experts=1, top_k=2
)
ds_moe_demo.train()

print("Bias before training steps:", ds_moe_demo.expert_bias.numpy().round(4))

# Run a few forward passes (bias adjusts automatically)
for step in range(20):
    x = torch.randn(2, 16, 32)
    _, _ = ds_moe_demo(x)

print("Bias after 20 steps:       ", ds_moe_demo.expert_bias.numpy().round(4))
print("\nNegative bias = overloaded expert (router will send fewer tokens)")
print("Positive bias = underloaded expert (router will send more tokens)")

Bias before training steps: [0. 0. 0. 0. 0. 0. 0. 0.]
Bias after 20 steps:        [-0.023  0.016 -0.004 -0.002 -0.001  0.001 -0.017  0.03 ]

Negative bias = overloaded expert (router will send fewer tokens)
Positive bias = underloaded expert (router will send more tokens)


In [23]:
class DeepSeekTransformerBlock(nn.Module):
    """Full DeepSeek-V3 Transformer block: MLA + DeepSeekMoE + RMSNorm."""

    def __init__(self, d_model, n_heads, kv_latent_dim, q_latent_dim,
                 rope_head_dim, expert_ff_dim, shared_ff_dim,
                 n_routed_experts=16, n_shared_experts=1, top_k=4):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = MultiHeadLatentAttention(
            d_model, n_heads, kv_latent_dim, q_latent_dim, rope_head_dim
        )
        self.norm2 = RMSNorm(d_model)
        self.moe = DeepSeekMoELayer(
            d_model, expert_ff_dim, shared_ff_dim,
            n_routed_experts, n_shared_experts, top_k
        )

    def forward(self, x):
        # Pre-norm (modern convention)
        x = x + self.attn(self.norm1(x))
        moe_out, router_logits = self.moe(self.norm2(x))
        x = x + moe_out
        return x, router_logits


block = DeepSeekTransformerBlock(
    d_model=128, n_heads=4, kv_latent_dim=32, q_latent_dim=64,
    rope_head_dim=16, expert_ff_dim=32, shared_ff_dim=64,
    n_routed_experts=16, top_k=4
)
x = torch.randn(1, 8, 128)
out, _ = block(x)
print(f"DeepSeek Block: {x.shape} → {out.shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")

DeepSeek Block: torch.Size([1, 8, 128]) → torch.Size([1, 8, 128])
Parameters: 278,784


## 6. Multi-Token Prediction (MTP)

Standard LLMs predict only the **next token**. DeepSeek-V3 predicts the next **D tokens**
simultaneously using additional prediction heads.

Each head $d$ predicts token $t + d$ given context up to $t$. The training loss:

$L = L_1 + \lambda \sum_{d=2}^{D} L_d$

**Why this helps:**
- Forces the model to plan ahead (not just pattern-match the next token)
- Provides richer gradient signal during training
- The extra heads are discarded at inference (no cost)
- DeepSeek uses $D = 2$ (predict next 2 tokens)

In [24]:
class MultiTokenPrediction(nn.Module):
    """Multi-Token Prediction heads (discarded at inference)."""

    def __init__(self, d_model, vocab_size, n_extra_heads=1):
        super().__init__()
        # Main head (always used)
        self.main_head = nn.Linear(d_model, vocab_size, bias=False)
        # Extra heads for future token prediction (training only)
        self.extra_heads = nn.ModuleList([
            nn.Sequential(
                RMSNorm(d_model),
                nn.Linear(d_model, d_model, bias=False),
                nn.SiLU(),
                nn.Linear(d_model, vocab_size, bias=False),
            )
            for _ in range(n_extra_heads)
        ])

    def forward(self, hidden_states, targets=None, mtp_lambda=0.3):
        """Returns main logits + MTP loss if targets provided."""
        main_logits = self.main_head(hidden_states)  # [B, N, vocab]

        if targets is None or not self.training:
            return main_logits, 0.0

        # Main loss: predict next token
        main_loss = F.cross_entropy(
            main_logits[:, :-1].reshape(-1, main_logits.size(-1)),
            targets[:, 1:].reshape(-1)
        )

        # Extra losses: predict future tokens
        mtp_loss = 0.0
        for d, head in enumerate(self.extra_heads, start=2):
            extra_logits = head(hidden_states)
            if targets.size(1) > d:
                mtp_loss += F.cross_entropy(
                    extra_logits[:, :-d].reshape(-1, extra_logits.size(-1)),
                    targets[:, d:].reshape(-1)
                )

        total_loss = main_loss + mtp_lambda * mtp_loss
        return main_logits, total_loss


# Demo
mtp = MultiTokenPrediction(d_model=64, vocab_size=100, n_extra_heads=1)
hidden = torch.randn(1, 16, 64)
targets = torch.randint(0, 100, (1, 16))

mtp.train()
logits, loss = mtp(hidden, targets)
print(f"MTP: hidden {hidden.shape} → logits {logits.shape}")
print(f"MTP loss (main + extra): {loss.item():.4f}")

mtp.eval()
logits, loss = mtp(hidden)  # no targets at inference
print(f"Inference: only main head, extra heads discarded, loss = {loss}")

MTP: hidden torch.Size([1, 16, 64]) → logits torch.Size([1, 16, 100])
MTP loss (main + extra): 6.4592
Inference: only main head, extra heads discarded, loss = 0.0


## 7. FP8 Mixed Precision Training

DeepSeek-V3 uses **FP8** (8-bit floating point) for most matrix multiplications during
training — not just inference. This was a key factor in the $5.5M training cost.

| Format | Bits | Sign | Exponent | Mantissa | Range | Precision |
|---|---|---|---|---|---|---|
| FP32 | 32 | 1 | 8 | 23 | $\pm 3.4 \times 10^{38}$ | ~7 digits |
| BF16 | 16 | 1 | 8 | 7 | $\pm 3.4 \times 10^{38}$ | ~2.4 digits |
| FP16 | 16 | 1 | 5 | 10 | $\pm 6.5 \times 10^{4}$ | ~3.3 digits |
| **E4M3** | **8** | 1 | 4 | 3 | $\pm 448$ | ~1 digit |
| **E5M2** | **8** | 1 | 5 | 2 | $\pm 57344$ | ~0.6 digits |

DeepSeek uses:
- **E4M3** for forward pass (more precision for activations)
- **E5M2** for backward pass (more range for gradients)
- **BF16/FP32** for master weights and optimizer states

See `03_Training_Techniques/02_Mixed_Precision_Training.ipynb` for FP16/BF16 fundamentals.

| Format | Bits | Exponent | Mantissa | Max Value | Use in DeepSeek-V3 | Memory/param |
|---|---|---|---|---|---|---|
| FP32 | 32 | 8 | 23 | 3.4e38 | Master weights | 4B |
| BF16 | 16 | 8 | 7 | 3.4e38 | Accumulation | 2B |
| FP16 | 16 | 5 | 10 | 6.5e4 | N/A | 2B |
| **FP8-E4M3** | **8** | 4 | 3 | 448 | **Forward matmuls** | **1B** |
| **FP8-E5M2** | **8** | 5 | 2 | 57344 | **Backward matmuls** | **1B** |

FP8 halves the memory and doubles throughput vs FP16 for matrix multiplications.
This is why DeepSeek-V3 cost $5.5M instead of ~$50M.

## 8. DualPipe Parallelism

Training a 671B model requires 2048+ GPUs. DeepSeek-V3 uses **DualPipe** — a novel
pipeline parallelism strategy:

Standard pipeline parallelism has "bubble time" — GPUs wait while micro-batches
propagate through the pipeline. DualPipe overlaps:
- Forward pass of micro-batch $B_{i+1}$ with backward pass of micro-batch $B_i$
- Communication of MoE expert activations with computation

Combined with:
- **Expert parallelism**: each GPU hosts a subset of the 256 experts
- **Data parallelism**: replicate attention layers across GPUs
- **Sequence parallelism**: split long sequences across GPUs

## 9. Training Recipe

| Phase | Details |
|---|---|
| **Pre-training** | 14.8T tokens, 2048 H800 GPUs, ~2 months |
| **Context extension** | 4K → 32K → 128K via YaRN-style RoPE scaling |
| **SFT** | 1.5M curated instruction-response pairs |
| **GRPO alignment** | Group Relative Policy Optimization (see `04_RL/03_GRPO`) |
| **Distillation** | V3 → DeepSeek-R1 distilled models (1.5B, 7B, 16B, 67B) |

**Training cost:** $5.5M total (pre-training only, on H800 GPUs at ~$2/GPU-hour).

Compare:
- LLaMA 3 405B: estimated ~$100M
- GPT-4: estimated ~$100M+
- DeepSeek-V3: **$5.5M** (18x cheaper than LLaMA 3)

| Model | Total Params | Training Tokens | Training Cost | Key Efficiency |
|---|---|---|---|---|
| GPT-4 (rumored) | ~1.8T | ~13T | ~$100M+ | N/A |
| LLaMA 3 405B | 405B | 15T | ~$100M | Dense, brute force |
| Gemini Ultra | ~1T | ~10T | ~$100M+ | Dense, TPU |
| **DeepSeek-V3** | **671B** | **14.8T** | **$5.5M** | **FP8 + MoE + DualPipe** |

## 10. Full Model Assembly

Let's assemble a scaled-down DeepSeek-V3 that combines all the components.

In [25]:
class DeepSeekV3(nn.Module):
    """
    Scaled-down DeepSeek-V3 architecture.
    Combines: MLA + DeepSeekMoE + RoPE + RMSNorm + SwiGLU + MTP.
    """

    def __init__(self, vocab_size=1000, d_model=128, n_heads=4,
                 kv_latent_dim=32, q_latent_dim=64, rope_head_dim=16,
                 expert_ff_dim=32, shared_ff_dim=64,
                 n_routed_experts=16, n_shared_experts=1, top_k=4,
                 n_layers=4, max_seq_len=512, mtp_heads=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            DeepSeekTransformerBlock(
                d_model, n_heads, kv_latent_dim, q_latent_dim,
                rope_head_dim, expert_ff_dim, shared_ff_dim,
                n_routed_experts, n_shared_experts, top_k
            )
            for _ in range(n_layers)
        ])

        self.norm = RMSNorm(d_model)
        self.mtp = MultiTokenPrediction(d_model, vocab_size, mtp_heads)

    def forward(self, input_ids, targets=None):
        x = self.embed(input_ids)

        all_router_logits = []
        for layer in self.layers:
            x, router_logits = layer(x)
            all_router_logits.append(router_logits)

        x = self.norm(x)
        logits, mtp_loss = self.mtp(x, targets)

        return logits, mtp_loss, all_router_logits


# Create mini DeepSeek-V3
model = DeepSeekV3(
    vocab_size=1000, d_model=128, n_heads=4,
    kv_latent_dim=32, q_latent_dim=64, rope_head_dim=16,
    expert_ff_dim=32, shared_ff_dim=64,
    n_routed_experts=16, n_shared_experts=1, top_k=4,
    n_layers=4, mtp_heads=1
)

tokens = torch.randint(0, 1000, (1, 16))
targets = torch.randint(0, 1000, (1, 16))

model.train()
logits, loss, router_logits = model(tokens, targets)

total_params = sum(p.numel() for p in model.parameters())
print(f"Mini DeepSeek-V3:")
print(f"  Layers: 4, Experts: 16 routed + 1 shared, Top-K: 4")
print(f"  Input:  {tokens.shape}")
print(f"  Output: {logits.shape}")
print(f"  MTP loss: {loss.item():.4f}")
print(f"  Total parameters: {total_params:,}")

Mini DeepSeek-V3:
  Layers: 4, Experts: 16 routed + 1 shared, Top-K: 4
  Input:  torch.Size([1, 16])
  Output: torch.Size([1, 16, 1000])
  MTP loss: 9.2227
  Total parameters: 1,515,776


In [26]:
# Architecture summary
print("=" * 60)
print("DeepSeek-V3 Architecture Summary (Mini Version)")
print("=" * 60)

embed_params = sum(p.numel() for p in model.embed.parameters())
layer_params = sum(p.numel() for p in model.layers[0].parameters())
mtp_params = sum(p.numel() for p in model.mtp.parameters())
attn_params = sum(p.numel() for p in model.layers[0].attn.parameters())
moe_params = sum(p.numel() for p in model.layers[0].moe.parameters())

print(f"\n{'Component':<30s} {'Params':>12s}")
print(f"{'-'*42}")
print(f"{'Embedding':<30s} {embed_params:>12,}")
print(f"{'MLA Attention (per layer)':<30s} {attn_params:>12,}")
print(f"{'DeepSeekMoE (per layer)':<30s} {moe_params:>12,}")
print(f"{'  - 16 routed experts':<30s} {sum(p.numel() for p in model.layers[0].moe.routed_experts.parameters()):>12,}")
print(f"{'  - 1 shared expert':<30s} {sum(p.numel() for p in model.layers[0].moe.shared_experts.parameters()):>12,}")
print(f"{'  - Router':<30s} {sum(p.numel() for p in model.layers[0].moe.gate.parameters()):>12,}")
print(f"{'MTP heads':<30s} {mtp_params:>12,}")
print(f"{'-'*42}")
print(f"{'TOTAL':<30s} {total_params:>12,}")

DeepSeek-V3 Architecture Summary (Mini Version)

Component                            Params
------------------------------------------
Embedding                           128,000
MLA Attention (per layer)            55,296
DeepSeekMoE (per layer)             223,232
  - 16 routed experts               196,608
  - 1 shared expert                  24,576
  - Router                            2,048
MTP heads                           272,512
------------------------------------------
TOTAL                             1,515,776


In [27]:
# Compare: mini DeepSeek-V3 vs a standard dense Transformer
dense_model = nn.Sequential(
    nn.Embedding(1000, 128),
    nn.TransformerEncoder(
        nn.TransformerEncoderLayer(d_model=128, nhead=4, dim_feedforward=512, batch_first=True),
        num_layers=4
    ),
    nn.Linear(128, 1000),
)

dense_params = sum(p.numel() for p in dense_model.parameters())

print(f"Parameter Comparison:")
print(f"  Dense Transformer:  {dense_params:>10,} params (all active per token)")
print(f"  Mini DeepSeek-V3:   {total_params:>10,} params (16 experts, top-4 active)")
print(f"  Ratio: DeepSeek has {total_params/dense_params:.1f}x more total params")
print(f"  But only ~{4/16:.0%} of MoE params active per token")

Parameter Comparison:
  Dense Transformer:   1,050,088 params (all active per token)
  Mini DeepSeek-V3:    1,515,776 params (16 experts, top-4 active)
  Ratio: DeepSeek has 1.4x more total params
  But only ~25% of MoE params active per token


## 11. Innovation Summary

| Innovation | What it does | Prerequisite Notebook |
|---|---|---|
| **MLA** | Compresses KV cache via low-rank projection | `05_Papers/03_Attention_Is_All_You_Need` |
| **DeepSeekMoE** | 256 fine-grained experts + shared experts | `05_Papers/04_Mixture_of_Experts` |
| **Aux-loss-free balancing** | Bias-based load balancing without hurting main loss | `05_Papers/04_Mixture_of_Experts` |
| **Decoupled RoPE** | Position encoding separate from KV content | `05_Papers/02_RoPE` |
| **Multi-Token Prediction** | Predict D future tokens for richer training signal | — (new in this notebook) |
| **FP8 training** | 8-bit matrix multiplications during training | `03_Training/02_Mixed_Precision` |
| **SwiGLU** | Gated activation replacing ReLU | — (standard in modern LLMs) |
| **RMSNorm** | Simpler normalization than LayerNorm | — (standard in modern LLMs) |
| **GRPO alignment** | RL without critic for preference learning | `04_RL/03_GRPO_Language_Models` |

## 12. Key Takeaways

1. **MLA compresses the KV cache by 93%.** By projecting K, V into a low-rank latent space, inference memory drops dramatically. Only the small latent vector is cached, not full K, V.

2. **256 fine-grained experts + shared experts outperform 8 large experts.** DeepSeek's MoE uses many small specialists + one generalist. More experts = more granular specialization.

3. **Auxiliary-loss-free balancing preserves model quality.** The bias trick avoids contaminating the training loss with load balancing objectives — a simple heuristic that works better than the standard approach.

4. **Multi-Token Prediction improves training signal.** Predicting 2+ tokens ahead forces the model to plan, providing richer gradients. The extra heads are free at inference.

5. **FP8 cuts training cost by ~2x.** 8-bit matrix multiplications halve memory and double throughput for the compute-heavy operations. This is the biggest factor in the $5.5M price tag.

6. **Open-source can compete with frontier labs.** DeepSeek-V3 matches GPT-4 level performance at 1/18th the cost, proving that algorithmic innovation matters more than raw compute.

### Further Reading

- DeepSeek-AI (2024). *DeepSeek-V3 Technical Report.* https://arxiv.org/abs/2412.19437
- DeepSeek-AI (2025). *DeepSeek-R1.* https://arxiv.org/abs/2501.12948
- DeepSeek-AI (2024). *DeepSeek-V2: A Strong, Economical, and Efficient MoE Language Model.* https://arxiv.org/abs/2405.04434